# Bài tập - Cứu bức ảnh siêu thị thiếu sáng
**BVU - Cao học - Xử lý ảnh - Trương Đình Phúc**

Đề bài: Ảnh camera siêu thị bị tối. Yêu cầu:
- (a) Kéo sáng bằng hiệu chỉnh gamma (kỹ thuật bảng tra LUT).
- (b) Cân bằng histogram theo hai cách rồi so màu: áp thẳng trên 3 kênh BGR so với áp trên kênh L của LAB.

**Đoán trước khi chạy:**
- Để kéo sáng vùng tối thì gamma nên < 1 hay > 1? -> **gamma < 1** (ví dụ 0.5) vì hàm `(i/255)**gamma` với gamma < 1 sẽ đẩy các giá trị nhỏ (vùng tối) lên cao hơn.
- Cân bằng ảnh màu trên 3 kênh BGR hay trên kênh L thì giữ được màu? -> **Cân bằng trên kênh L của LAB** giữ được màu gốc, vì kênh L chỉ mang thông tin độ sáng, còn A/B (màu sắc) không bị đổi. Cân bằng riêng từng kênh B, G, R sẽ làm lệch tỉ lệ màu giữa các kênh -> ảnh bị ám màu (sai màu).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install opencv-python-headless matplotlib numpy

## Hàm đọc và hiển thị ảnh (dùng chung cho các cell bên dưới)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

FOLDER = "/content/drive/MyDrive/Colab Notebooks"

def load(name):
    """Tìm ảnh 'name' trong FOLDER với nhiều đuôi file khác nhau."""
    extensions = [".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp"]
    for ext in extensions:
        path = os.path.join(FOLDER, name + ext)
        if os.path.exists(path):
            img = cv2.imread(path)
            if img is not None:
                print("Đã đọc ảnh:", path)
                return img
    raise FileNotFoundError(f"Không tìm thấy ảnh '{name}' trong thư mục {FOLDER}")

def show(items, title):
    n = len(items)
    plt.figure(figsize=(5 * n, 5))
    plt.suptitle(title, fontsize=15)
    for i, (name, image) in enumerate(items, start=1):
        plt.subplot(1, n, i)
        if image.ndim == 2:
            plt.imshow(image, cmap="gray")
        else:
            plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plt.title(name)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## (a) Kéo sáng bằng hiệu chỉnh gamma (LUT)

In [ ]:
img = load("sieuthi")

gamma = 0.5  # gamma < 1 de keo sang vung toi

lut = np.array(
    [((i / 255.0) ** gamma) * 255 for i in range(256)],
    dtype=np.uint8
)

sang = cv2.LUT(img, lut)

print(
    'Do sang trung binh: goc = %.0f -> sau gamma = %.0f'
    % (img.mean(), sang.mean())
)

show(
    [('Goc (toi)', img), ('Sau gamma (gamma=0.5)', sang)],
    'Bai 1a - Gamma keo sang'
)

## (b) Cân bằng histogram: 2 cách
- **Cách sai**: cân bằng histogram riêng lẻ trên từng kênh B, G, R rồi ghép lại -> làm lệch tỉ lệ giữa các kênh màu, ảnh bị ám màu.
- **Cách đúng**: chuyển sang không gian màu LAB, chỉ cân bằng kênh L (độ sáng), giữ nguyên A, B (màu sắc) rồi chuyển ngược lại BGR.

In [ ]:
# Cach SAI: can bang tung kenh BGR
he_sai = cv2.merge(
    [cv2.equalizeHist(channel) for channel in cv2.split(img)]
)

# Cach DUNG: can bang kenh L cua LAB
lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
L, A, B = cv2.split(lab)
L_eq = cv2.equalizeHist(L)
lab_eq = cv2.merge([L_eq, A, B])
he_dung = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

show(
    [
        ('Goc', img),
        ('HE tren BGR (SAI - le mau)', he_sai),
        ('HE tren kenh L cua LAB (DUNG)', he_dung)
    ],
    'Bai 1b - Can bang histogram: sai cach vs dung cach'
)

## Đáp án trắc nghiệm

| Câu | Chọn |
|---|---|
| (a) Gamma nên chọn | **0.5** (< 1, để kéo sáng vùng tối) |
| (b) Cân bằng histogram giữ màu tốt hơn khi áp trên kênh | **LAB** (kênh L) |

**Giải thích:**
- Gamma correction: `s = (r/255)^gamma * 255`. Với gamma < 1, các giá trị pixel nhỏ (tối) được ánh xạ lên các giá trị lớn hơn tương đối nhiều hơn so với các giá trị pixel lớn (sáng) -> làm sáng vùng tối mà không làm cháy sáng vùng đã sáng.
- Cân bằng histogram trên từng kênh B, G, R độc lập làm thay đổi tỉ lệ tương đối giữa 3 kênh màu tại mỗi điểm ảnh -> lệch màu (color shift). Trong khi đó, không gian LAB tách biệt độ sáng (L) khỏi thông tin màu (A, B), nên chỉ cân bằng L sẽ cải thiện độ tương phản mà giữ nguyên màu sắc gốc của ảnh.